# Decorators example
## Retry & Validate

In this notebook we build two practical, parametrized decorators and stack them together.
The goal is to see how decorators let us compose **different concerns** (like retrying on failure and validating inputs) without touching the function body.

**Prerequisites:** closures, `functools.wraps`, decorator factories, stacking
(see [Functions (advanced)](../12_functions_advanced.ipynb)).

In [ ]:
from functools import wraps
import time
import random

## 1. The `@retry` decorator

Network calls fail, file I/O can be flaky, external services go down.
Instead of writing a try/except loop every time we call an unreliable function, we can wrap it once with a decorator.

Let's start with a function that simulates a flaky service:

In [ ]:
import random

random.seed(None)


def flaky_function():
    """Simulate an unreliable network call."""
    if random.random() < 0.6:
        raise ConnectionError("Server not responding")
    return {"status": "ok", "data": 42}

Try calling it a few times — sometimes it works, sometimes it doesn't:

In [ ]:
for i in range(5):
    try:
        result = flaky_function()
        print(f"Call {i + 1}: {result}")
    except ConnectionError as e:
        print(f"Call {i + 1}: FAILED — {e}")

We *could* wrap every call site in a retry loop, but that quickly becomes tedious and clutters our code.
Let's build a decorator instead.

### First version: simple retry (hard-coded)

In [ ]:
def retry_simple(fn):
    """Retry a function up to 3 times with a 0.5s delay."""

    @wraps(fn)
    def wrapper(*args, **kwargs):
        last_exception = None
        for attempt in range(1, 4):
            try:
                return fn(*args, **kwargs)
            except Exception as e:
                last_exception = e
                print(f"  Attempt {attempt} failed: {e}")
                time.sleep(0.5)
        raise last_exception

    return wrapper


@retry_simple
def flaky_function():
    if random.random() < 0.6:
        raise ConnectionError("Server not responding")
    return {"status": "ok", "data": 42}


flaky_function()

This works, but the retry count, delay, and which exceptions to catch are all hard-coded.
Let's make it configurable — we need a **decorator factory**.

### Parametrized `@retry`

In [ ]:
def retry(max_attempts=3, delay=0.5, exceptions=(Exception,)):
    """
    Decorator factory: retry a function up to `max_attempts` times,
    waiting `delay` seconds between attempts.
    Only catches exceptions listed in `exceptions`.
    """

    def decorator(fn):
        @wraps(fn)
        def wrapper(*args, **kwargs):
            last_exception = None
            for attempt in range(1, max_attempts + 1):
                try:
                    return fn(*args, **kwargs)
                except exceptions as e:
                    last_exception = e
                    print(f"  [{fn.__name__}] Attempt {attempt}/{max_attempts} failed: {e}")
                    if attempt < max_attempts:
                        time.sleep(delay)
            raise last_exception

        return wrapper

    return decorator

Let's try it out:

In [ ]:
@retry(max_attempts=5, delay=0.3, exceptions=(ConnectionError,))
def flaky_function():
    if random.random() < 0.6:
        raise ConnectionError("Server not responding")
    return {"status": "ok", "data": 42}


flaky_function()

Notice how only `ConnectionError` is caught.
If the function raises something else, the decorator lets it propagate immediately — no retries wasted:

In [ ]:
@retry(max_attempts=5, delay=0.3, exceptions=(ConnectionError,))
def broken_function():
    raise ValueError("This is a bug, not a transient failure")


try:
    broken_function()
except ValueError as e:
    print(f"ValueError propagated immediately: {e}")

## 2. The `@validate_types` decorator

Python is dynamically typed, which is flexible but can lead to confusing errors deep inside a function.
A type-checking decorator catches bad inputs early, at the "front door."

Consider a simple calculation:

In [ ]:
def compute(x, y):
    return x * y + 1


# Works fine with numbers:
print(compute(3, 4.5))

# But with wrong types, the error is confusing:
try:
    compute("hello", 4.5)
except TypeError as e:
    print(f"Confusing error: {e}")

Let's build a decorator that checks types before the function runs.

To keep things simple, our decorator will require the function to be called with **keyword arguments**.
This way we can match argument names directly — no introspection needed:

In [ ]:
def validate_types(**expected_types):
    """
    Decorator factory: check that keyword arguments match the specified types.

    Usage: @validate_types(x=int, y=float)
    """

    def decorator(fn):
        @wraps(fn)
        def wrapper(**kwargs):
            for param_name, expected_type in expected_types.items():
                if param_name in kwargs:
                    value = kwargs[param_name]
                    if not isinstance(value, expected_type):
                        raise TypeError(
                            f"Argument '{param_name}' must be {expected_type.__name__}, "
                            f"got {type(value).__name__}: {value!r}"
                        )
            return fn(**kwargs)

        return wrapper

    return decorator

Let's try it:

In [ ]:
@validate_types(x=int, y=(int, float))
def compute(x, y):
    return x * y + 1


# Correct types — works fine:
print(compute(x=3, y=4.5))

# Wrong types — clear error message:
try:
    compute(x="hello", y=4.5)
except TypeError as e:
    print(f"Clear error: {e}")

The function body never changed.
The validation concern is completely **separate** from the business logic.

> **Note:** requiring keyword arguments is a simplification. A production-grade version would use
> `inspect.signature` to handle positional arguments too — but that's beyond our scope here.
> In practice, you'd use a library like [`pydantic`](https://docs.pydantic.dev/) for this.

### Advanced alternative: handling positional arguments

If you want `validate_types` to work with **both** positional and keyword arguments,
you can use `inspect.signature` to bind them to parameter names:

In [ ]:
import inspect


def validate_types_advanced(**expected_types):
    """Version that handles both positional and keyword arguments."""

    def decorator(fn):
        sig = inspect.signature(fn)

        @wraps(fn)
        def wrapper(*args, **kwargs):
            bound = sig.bind(*args, **kwargs)
            bound.apply_defaults()

            for param_name, expected_type in expected_types.items():
                if param_name in bound.arguments:
                    value = bound.arguments[param_name]
                    if not isinstance(value, expected_type):
                        raise TypeError(
                            f"Argument '{param_name}' must be {expected_type.__name__}, "
                            f"got {type(value).__name__}: {value!r}"
                        )
            return fn(*args, **kwargs)

        return wrapper

    return decorator


# Now positional calls work too:
@validate_types_advanced(x=int, y=(int, float))
def compute(x, y):
    return x * y + 1


print(compute(3, 4.5))  # positional argumnets work!

compute("hello", y=4.5)

## 3. Stacking them together

Now the real power: combining both decorators on a single function.

Imagine we have a flaky sensor API. We want to:
1. **Validate** that we're passing the right types
2. **Retry** if the sensor doesn't respond

The stacking order matters! The decorator **closest to `def`** wraps first (its wrapper is the **innermost** layer):

```python
@retry(...)          # outer: retries the validated call
@validate_types(...) # inner: validates first, then calls the function
def my_function(...):
    ...
```

In [ ]:
@retry(max_attempts=4, delay=0.3, exceptions=(ConnectionError,))
@validate_types(sensor_id=int, threshold=(int, float))
def fetch_sensor_reading(sensor_id, threshold):
    """Fetch a reading from a flaky sensor API."""
    if random.random() < 0.5:
        raise ConnectionError(f"Sensor {sensor_id} not responding")
    reading = random.uniform(0, 100)
    return round(reading, 2) if reading > threshold else 0.0

Call with correct types — retries happen on `ConnectionError`:

In [ ]:
result = fetch_sensor_reading(sensor_id=42, threshold=10.0)
print(f"Reading: {result}")

Call with wrong types — `TypeError` is raised **immediately**, no retries attempted
(because `TypeError` is not in the retry `exceptions` tuple):

In [ ]:
try:
    fetch_sensor_reading(sensor_id="abc", threshold=10.0)
except TypeError as e:
    print(f"Caught immediately: {e}")

This is exactly the point: each decorator handles **one concern**, and stacking them composes behavior cleanly.

## Recap

- **Decorator factories** use three nested functions: factory → decorator → wrapper.
- Always use `functools.wraps` to preserve the original function's name and docstring.
- **Stacking order matters:** the decorator closest to `def` wraps first (runs first when the function is called).
- These patterns are used in well-known libraries. Use these and do not reivent the well for production code!
  - Retry → [`tenacity`](https://tenacity.readthedocs.io/)
  - Type validation → [`pydantic`](https://docs.pydantic.dev/)
